# 04 — ADME Noise Injection Study (Phase 1: Coarse Scan)

**Goal**: Quantify how RF / LightGBM / FCNN performance degrades under three types of
label noise (Gaussian, systematic bias, gross errors — Landrum & Riniker taxonomy), on the
ADME `hybrid` featureset (FCFP4 + normalized RDKit2D descriptors), for the HLM and SOL
endpoints. See `DECISIONS.md` ADR-005 for the exact noise formulations.

**Featureset note**: this notebook originally ran on `fcfp4` only. Those results are
preserved as `noise_scan_results_fcfp4.csv` / `noise_scan_predictions_fcfp4.pkl` /
`figures/noise_scan_coarse_fcfp4.png`. Checkpoint and figure filenames are now suffixed by
`ANALYSIS_FEATURESET` (set in §2) so different featuresets never collide or silently overwrite each
other's results.

**Design** (mirrors `03_adme_data_quantity.ipynb`'s coarse-then-zoom methodology, see
conversation record / ADR-005 for full rationale):
1. **Phase 1 (this notebook)** — coarse scan: fixed training fraction (1.0, full N), the
   full ADR-005 level grid per noise type, few seeds, baseline (default hyperparameter) arm
   only. Purpose is purely to locate each `(endpoint, model, noise_type)` curve's collapse
   point, not to produce a final publishable curve. The extreme level of each grid
   (`sigma_frac`/`bias_frac` = 1.0) is a sanity-check anchor — it should land near the
   `NoiseEstimator` theoretical ceiling (R² ≈ 0), not a claimed-realistic operating point.
2. **Phase 2** (later) — identify the collapse point per `(endpoint, model, noise_type)`:
   the same fixed-threshold rule used in `03`'s §5, applied along the noise axis instead of
   the fraction axis.
3. **Phase 3** (later) — zoom in below the identified knee with a finer noise-level grid and
   more seeds, producing the reported degradation curves.

**Metrics**: R² (primary) and MAE (secondary), same choice as `03` — see that notebook's
title cell for the rationale. Pearson r / Spearman / CCC are also recorded (via
`evaluate_model`) but not plotted here.

**Fraction fixed at 1.0**: unlike `03`, this notebook does not vary training-set size — all
fits use the full training pool so that noise is the only manipulated variable. The results
schema still carries a `fraction` column (fixed to `1.0`) purely so the checkpoint schema
stays identical to `03`'s, per the schema-reuse note in that notebook's §2.

## 0 — Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.preprocessing import RobustScaler

from src.models import get_paper_models, evaluate_model, run_checkpointed_eval
from src.noise import add_gaussian_noise, add_systematic_bias, add_gross_errors

SEED = 42
DATA_PROC = '../data/processed'
FIGURES = '../figures'
os.makedirs(FIGURES, exist_ok=True)

print('Imports OK')

## 1 — Load existing hybrid splits

Reuses the same `(endpoint, featureset)` splits as `03_adme_data_quantity.ipynb` §1 — no
refeaturization. Each entry carries `X_train`/`X_test` (raw hybrid features — FCFP4 bits +
normalized RDKit2D descriptors concatenated — for RF/LightGBM) and `y_train`/`y_test`. FCNN
also uses raw `X_train`/`X_test` here — its `RobustScaler` is fit fresh on the full training
pool in §3 below (fraction is fixed at 1.0 in this notebook, so there is no per-subsample
refit needed).

In [ ]:
ANALYSIS_FEATURESET = 'hybrid'
EPS = ['HLM', 'SOL']

splits = joblib.load(f'{DATA_PROC}/section4_splits.pkl')
ep_data = {ep: splits[(ep, ANALYSIS_FEATURESET)] for ep in EPS}

for ep in EPS:
    d = ep_data[ep]
    print(f"  {ep}: X_train={d['X_train'].shape}, X_test={d['X_test'].shape}")

## 2 — Config

In [ ]:
FRACTION = 1.0  # fixed -- noise is the only manipulated variable in this notebook
N_SEEDS_COARSE = 5
MODELS = ['RF', 'LightGBM', 'FCNN']
SCALED_MODELS = {'FCNN'}  # models that need RobustScaler'd X

ARM = 'base'  # untuned -- Phase 1 is baseline-only, no tuning arm here

# Coarse-scan levels per ADR-005 (DECISIONS.md). sigma_frac/bias_frac and error_frac are
# fractions of std(y) / N respectively -- 0.0 is covered once by the shared 'none' baseline
# below rather than once per noise type (all three collapse to the same unnoised fit at
# level 0, so repeating it three times would waste fits without adding information).
NOISE_GRID = {
    'gaussian':         [0.1, 0.3, 0.5, 1.0],
    'systematic_bias':  [0.1, 0.3, 0.5, 1.0],
    'gross_errors':     [0.01, 0.05, 0.10, 0.20],
}

n_baseline = len(EPS) * len(MODELS) * N_SEEDS_COARSE
n_noise = len(EPS) * len(MODELS) * sum(len(v) for v in NOISE_GRID.values()) * N_SEEDS_COARSE
print(f'{n_baseline} baseline fits + {n_noise} noise fits = {n_baseline + n_noise} total fits')

## 3 — Coarse sweep

Checkpointed via `run_checkpointed_eval` — re-running this cell only computes missing
`(endpoint, model, fraction, seed, noise_type, noise_level, arm)` keys, so it's safe to
interrupt or extend `NOISE_GRID`/`N_SEEDS_COARSE` later without recomputing everything. The
`KEY_COLS` order matches `03_adme_data_quantity.ipynb` exactly — this is the schema reuse
anticipated in that notebook's §2 config cell.

In [ ]:
RESULTS_CSV = f'{DATA_PROC}/noise_scan_results_{ANALYSIS_FEATURESET}.csv'
PREDICTIONS_PKL = f'{DATA_PROC}/noise_scan_predictions_{ANALYSIS_FEATURESET}.pkl'
KEY_COLS = ('endpoint', 'model', 'fraction', 'seed', 'noise_type', 'noise_level', 'arm')

paper_models = get_paper_models()

NOISE_FNS = {
    'gaussian':        lambda y, level, seed: add_gaussian_noise(y, sigma_frac=level, random_state=seed),
    'systematic_bias': lambda y, level, seed: add_systematic_bias(y, bias_frac=level, random_state=seed),
    'gross_errors':    lambda y, level, seed: add_gross_errors(y, error_frac=level, random_state=seed),
}

def compute_one(key):
    ep, model_name, frac, seed, noise_type, noise_level, arm = key
    d = ep_data[ep]
    scaled = model_name in SCALED_MODELS
    X_train, X_test = d['X_train'], d['X_test']
    y_train, y_test = d['y_train'], d['y_test']

    if noise_type == 'none':
        y_noisy = y_train
    else:
        y_noisy = NOISE_FNS[noise_type](y_train, noise_level, seed)

    X_sub = X_train
    if scaled:
        scaler = RobustScaler().fit(X_sub)
        X_sub = scaler.transform(X_sub)
        X_test = scaler.transform(X_test)

    model = clone(paper_models[model_name])
    model.fit(X_sub, y_noisy)
    y_pred = model.predict(X_test)
    metrics = evaluate_model(None, None, y_test, y_pred=y_pred)

    row = {
        'endpoint': ep, 'model': model_name, 'fraction': frac, 'n_train': len(X_sub),
        'seed': seed, 'noise_type': noise_type, 'noise_level': noise_level, 'arm': arm,
        **metrics,
    }
    pred = {'y_test': y_test, 'y_pred_test': y_pred}
    return row, pred

baseline_keys = [
    (ep, model_name, FRACTION, seed, 'none', 0.0, ARM)
    for ep in EPS
    for model_name in MODELS
    for seed in range(N_SEEDS_COARSE)
]
noise_keys = [
    (ep, model_name, FRACTION, seed, noise_type, level, ARM)
    for ep in EPS
    for model_name in MODELS
    for noise_type, levels in NOISE_GRID.items()
    for level in levels
    for seed in range(N_SEEDS_COARSE)
]
keys = baseline_keys + noise_keys

scan_results, scan_predictions = run_checkpointed_eval(
    keys, compute_one, RESULTS_CSV, PREDICTIONS_PKL, key_cols=KEY_COLS,
)
print(scan_results.shape)
scan_results.head()

## 4 — Plot: R² and MAE vs noise level

In [ ]:
MODEL_COLORS = {'RF': '#1f77b4', 'LightGBM': '#d62728', 'FCNN': '#ff9900'}
NOISE_TYPES = ['gaussian', 'systematic_bias', 'gross_errors']

fig, axes = plt.subplots(len(NOISE_TYPES), len(EPS), figsize=(6 * len(EPS), 4 * len(NOISE_TYPES)),
                          sharex=False)

for row_i, noise_type in enumerate(NOISE_TYPES):
    baseline_df = scan_results[scan_results['noise_type'] == 'none'].copy()
    baseline_df['noise_level'] = 0.0
    type_df = scan_results[scan_results['noise_type'] == noise_type]
    plot_df = pd.concat([baseline_df, type_df], ignore_index=True)

    for col, ep in enumerate(EPS):
        ep_df = plot_df[plot_df['endpoint'] == ep]
        ax = axes[row_i, col]

        for model_name in MODELS:
            m_df = ep_df[ep_df['model'] == model_name].groupby('noise_level').agg(
                R2_mean=('R2', 'mean'), R2_std=('R2', 'std'), n_seeds=('R2', 'size'),
            ).reset_index().sort_values('noise_level')
            m_df['R2_sem'] = m_df['R2_std'] / np.sqrt(m_df['n_seeds'])

            x = m_df['noise_level']
            color = MODEL_COLORS[model_name]

            # Wide/light band = std (seed-to-seed spread). Narrow/dark band = SEM (mean uncertainty).
            ax.plot(x, m_df['R2_mean'], marker='o', label=model_name, color=color)
            ax.fill_between(x, m_df['R2_mean'] - m_df['R2_std'], m_df['R2_mean'] + m_df['R2_std'],
                            alpha=0.12, color=color)
            ax.fill_between(x, m_df['R2_mean'] - m_df['R2_sem'], m_df['R2_mean'] + m_df['R2_sem'],
                            alpha=0.35, color=color)

        ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
        ax.set_title(f'{ep} -- {noise_type}')
        ax.set_ylabel('R² (test)')
        ax.set_xlabel(f'noise level ({noise_type})')
        ax.grid(alpha=0.3)

from matplotlib.patches import Patch
band_handles = [
    Patch(facecolor='grey', alpha=0.35, label='± SEM (mean uncertainty)'),
    Patch(facecolor='grey', alpha=0.12, label='± std (seed-to-seed spread)'),
]
model_handles, model_labels = axes[0, 0].get_legend_handles_labels()
axes[0, 0].legend(handles=model_handles + band_handles, fontsize=8)

fig.suptitle(f'Noise injection coarse scan -- RF / LightGBM / FCNN on {ANALYSIS_FEATURESET}', fontsize=12)
plt.tight_layout()
plt.savefig(f'{FIGURES}/noise_scan_coarse_{ANALYSIS_FEATURESET}.png', dpi=150, bbox_inches='tight')
plt.show()